In [1]:
import pandas as pd
import numpy as np

# Data Ingestion

In [2]:
df_mock = pd.read_csv("../data/mock/debit_dummy_data_20250705.csv")

In [3]:
df_mock['Transaction Datetime'] = pd.to_datetime(df_mock['Transaction Datetime'])

In [4]:
df_mock['CustomerSex'] = df_mock['CustomerSex'].where(df_mock['CustomerSex'].isin({"0", "1", "2"}), "N")
df_mock['CustomerSex'] = df_mock['CustomerSex'].fillna("N")

df_mock['HighRiskCustomer'] = df_mock['HighRiskCustomer'].where(df_mock['HighRiskCustomer'].isin({"0", "N"}), "N")
df_mock['HighRiskCustomer'] = df_mock['HighRiskCustomer'].fillna("N")

df_mock['CustomerAge'] = df_mock['CustomerAge'].fillna(0)

# Preprocessing

In [5]:
def is_convertible_to_float(x):
    try:
        float(x)
        return True
    except (ValueError, TypeError):
        return False

for col in ["MCC", "Country Code", "Currency Code", "POSMode"]:
    df_mock[col] = df_mock[col].astype(str)
    if col == "POSMode":
        df_mock[col] = df_mock[col].where(
            df_mock[col].apply(is_convertible_to_float), "__missing__"
        )
        df_mock[col] = df_mock[col].replace(
            to_replace=r"(?i)^nan$", value="__missing__", regex=True
        )
        df_mock[col] = df_mock[col].fillna("__missing__")
    else:
        df_mock[col] = df_mock[col].where(
            df_mock[col].apply(is_convertible_to_float), "-999"
        )
        df_mock[col] = df_mock[col].replace(
            to_replace=r"(?i)^nan$", value="-999", regex=True
        )
        df_mock[col] = df_mock[col].fillna("-999")

# Feature Engineering

In [6]:
import sys
import os
import warnings

warnings.filterwarnings("ignore")

# Get the absolute path to the project root (one level up from 'test')
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(project_root)

In [7]:
# functionalities
from src.calculation_features import (
    generate_rolling_features,
    label_risk_level_category,
    calculate_time_differences,
    calculate_rolling_txn_hour,
    create_ratio_features
)

In [8]:
# configs
from src.debit_card_config import (
    time_shift_config,
    time_windows,
    freq_config,
    dynamic_high_risk_config,
    duration_since_first_trnx_config,
    unique_count_config,
    monetary_config_1,
    monetary_config_2,
    monetary_config_3,
)
all_monetary_configs = (
    monetary_config_1
    + monetary_config_2
    + monetary_config_3
)

## Time (Transaction Hour) Features

In [10]:
df_txn_hour = calculate_rolling_txn_hour(
    df=df_mock,
    group_col="Debit_No",
    datetime_col="Transaction Datetime",
    windows=time_windows,
)

Processing TrnxHour Rolling Avg: 100%|██████████| 6/6 [00:00<00:00, 460.35it/s]


In [11]:
df_txn_hour = create_ratio_features(df_txn_hour, df_txn_hour.columns, "L15min", "L30D")

## RFM (Recency, Frequency, Monetary) Features

### Recency

#### Transaction Time Difference

In [12]:
df_time_diff = calculate_time_differences(
    df=df_mock,
    datetime_col="Transaction Datetime",
    groupby_col="Debit_No",
    time_window=time_windows,
    config=time_shift_config,
)

Calculating rolling averages: 100%|██████████| 2/2 [00:00<00:00, 328.69it/s]


#### Time Duration Since First to Current Transaction

In [13]:
df_time_firsttxn_to_current = generate_rolling_features(
    df=df_mock,
    datetime_col="Transaction Datetime",
    key_col="Transaction Serial No",
    features_config=duration_since_first_trnx_config,
)

Feature Config Progress: 100%|██████████| 2/2 [00:00<00:00, 40.83it/s]


In [14]:
for groupby_col in ["MCC", "Country Code"]:
    df_time_firsttxn_to_current = df_time_firsttxn_to_current.sort_values(by=["Debit_No", "Transaction Datetime"])

    # get first transaction time per target group within each primary group
    df_time_firsttxn_to_current[f"FirstTxnBy{groupby_col.replace(" ","")}"] = df_time_firsttxn_to_current.groupby(["Debit_No", groupby_col])[
        "Transaction Datetime"
    ].transform("first")

    # calculate duration in minutes
    amount_col = f"TimeFirstTxnToCurrent{groupby_col.replace(" ","")}"
    df_time_firsttxn_to_current[amount_col] = (
        df_time_firsttxn_to_current["Transaction Datetime"] - df_time_firsttxn_to_current[f"FirstTxnBy{groupby_col.replace(" ","")}"]
    ).dt.total_seconds() / 60

    df_time_firsttxn_to_current = df_time_firsttxn_to_current.drop(f"FirstTxnBy{groupby_col.replace(" ","")}", axis=1)

In [15]:
# ratio
df_time_firsttxn_to_current = create_ratio_features(
    df_time_firsttxn_to_current,
    df_time_firsttxn_to_current.columns,
    "L15min",
    "L30D"
)

### Frequency

#### Transaction Count

In [16]:
df_freq = generate_rolling_features(
    df=df_mock,
    datetime_col="Transaction Datetime",
    key_col="Transaction Serial No",
    features_config=freq_config,
)

Feature Config Progress: 100%|██████████| 4/4 [00:00<00:00, 64.17it/s]


In [17]:
# ratio
df_freq = create_ratio_features(
    df_freq,
    df_freq.columns,
    "L15min",
    "L30D"
)

#### Category Unique Count

In [18]:
df_mock["MCC Num"], uniques = df_mock["MCC"].factorize()
df_mock["Debit_No Num"], uniques = df_mock["Debit_No"].factorize()

df_unique_count = generate_rolling_features(
    df=df_mock,
    datetime_col="Transaction Datetime",
    key_col="Transaction Serial No",
    features_config=unique_count_config,
)

Feature Config Progress: 100%|██████████| 3/3 [00:00<00:00, 49.92it/s]


In [19]:
# ratio
df_unique_count = create_ratio_features(
    df_unique_count,
    df_unique_count.columns,
    "L15min",
    "L30D"
)

### Monetary

In [20]:
df_monetary = generate_rolling_features(
    df=df_mock,
    datetime_col="Transaction Datetime",
    key_col="Transaction Serial No",
    features_config=all_monetary_configs,
)

Feature Config Progress: 100%|██████████| 9/9 [00:00<00:00, 67.12it/s]


In [21]:
# ratio
df_monetary = create_ratio_features(
    df_monetary,
    df_monetary.columns,
    "L15min",
    "L30D"
)

# Join All Data

In [22]:
# Define the original columns
og_cols = df_mock.columns.tolist()

# Define the common keys for merging
merge_keys = ["Transaction Serial No", "Debit_No"]

# Extract columns
time_shift_cols = list(time_shift_config.keys())
freq_cols = [col for col in df_freq.columns if col not in og_cols]
unique_count_cols = [col for col in df_unique_count.columns if col not in og_cols]
monetary_cols = [col for col in df_monetary.columns if col not in og_cols]
time_firsttxn_to_current_cols = [
    col
    for col in df_time_firsttxn_to_current.columns
    if "TimeFirstTxn" in col
]
txn_hour_cols = [
    col
    for col in df_txn_hour.columns
    if "TxnHour" in col
]
# high_risk_category_col = [
#     col
#     for col in df_high_risk_label.columns
#     if f"IsTop{dynamic_high_risk_config["top_n"]}HighRisk" in col
# ]

In [23]:
from functools import reduce

# Subset dataframes
df_time_diff = df_time_diff[merge_keys + time_shift_cols]
df_freq = df_freq[merge_keys + freq_cols]
df_monetary = df_monetary[merge_keys + monetary_cols]
df_unique_count = df_unique_count[merge_keys + unique_count_cols]
df_time_firsttxn_to_current = df_time_firsttxn_to_current[
    merge_keys + time_firsttxn_to_current_cols
]
df_txn_hour = df_txn_hour[
    merge_keys + txn_hour_cols
]
# df_high_risk_label = df_high_risk_label[merge_keys + high_risk_category_col]

# Merge all feature dataframes
dfs_to_merge = [
    df_time_diff,
    df_freq,
    df_monetary,
    df_unique_count,
    df_time_firsttxn_to_current,
    df_txn_hour,
    # df_high_risk_label,
]

# Final join
df_mock_join = reduce(
    lambda left, right: pd.merge(left, right, on=merge_keys, how="outer"), dfs_to_merge
)

In [24]:
# Merge with additional features
df_mock_final = df_mock.merge(
    df_mock_join,
    on=merge_keys,
    how="left",
)

# Selecting Features and Saving Dataframe

In [25]:
final_derived_feats = [
    # "Transaction Amount",
    # "Customer Age",
    # "HighRiskCustomer",
    # "POSMode",
    # "CountTrxTrf",
    # "TotalTrxAmount10Mi",
    # "TotalTrxAmountL5min",
    # "TotalTrxAmount15Mi",
    # "TotalTrxAmountL1D",
    # "RatioTrxAmountL1DL15min",
    # "RatioTrxAmountL1DL10min",
    # "RatioTrxAmountL1DL5min",
    "SumAmtL30D",
    "TxnCountL30D",
    "CntUniqueCardNoByMCCL30D",
    "MaxAmtL30D",
    "AvgAmtL30D",
    "RatioCntUniqueCardNoByMCCL30DL15min",
    "RatioTxnCountL30DL15min",
    "AvgTxnHourL30D",
    "TxnTimeDifference",
    "TimeFirstTxnToCurrentMCC",
    "AvgTimeFirstTxnToCurrentMCCL30D",
    "AvgTxnHourL15min",
    "CntUniqueCardNoByMCCL15min",
]

In [26]:
for col in final_derived_feats:
    df_mock_final[col] = df_mock_final[col].fillna(0)

In [27]:
df_mock_final[
    og_cols + final_derived_feats    
].to_csv("../data/mock/debit_dummy_data_result_20250705.csv")

In [28]:
df_mock_final[
    og_cols + final_derived_feats    
].to_clipboard()